# Notebook 03 — Modelos Baseline

## Objetivo

Entrenar modelos clásicos de clasificación supervisada sobre representaciones BoW y TF-IDF.

**Experimento principal:** subconjunto político.  
**Experimento complementario:** dataset completo (solo baselines).

> El modelo aprende patrones lingüísticos del dataset; no verifica hechos.

**Métrica principal:** F2-score de la clase fake (prioriza recall de fake sin ignorar precisión).

In [23]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.paths import *
from src.plotting import setup_style, save_figure

setup_style()

from itertools import product
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_curve, auc
import joblib
from tqdm.auto import tqdm

from src.metrics import compute_metrics, metrics_to_row
from src.modeling import MODEL_CONFIGS, TEXT_FIELDS, STOPWORD_OPTS, build_model, build_vectorizer, get_text_col
from src.plotting import plot_confusion_matrix


## 1. Carga de splits

In [14]:

def load_splits(prefix):
    train = pd.read_csv(DATA_PROCESSED / f'{prefix}_train.csv')
    val = pd.read_csv(DATA_PROCESSED / f'{prefix}_val.csv')
    test = pd.read_csv(DATA_PROCESSED / f'{prefix}_test.csv')
    return train, val, test

pol_train, pol_val, pol_test = load_splits('politics')
full_train, full_val, full_test = load_splits('full')


## 2. Configuración de la grilla de experimentos

In [15]:

NGRAM_OPTS = [(1, 1), (1, 2)]
MAX_FEATURES_OPTS = [10000, 30000, 50000]
VECTORIZER_TYPES = ['bow', 'tfidf']


## 3. Entrenamiento y selección por validación (F2 fake)

In [25]:

def run_baseline_grid(train, val, dataset_scope, max_combos=None):
    """Entrena la grilla y elige hiperparámetros solo con validación."""
    results = []
    combos = list(product(
        TEXT_FIELDS.keys(), STOPWORD_OPTS.keys(), NGRAM_OPTS,
        MAX_FEATURES_OPTS, VECTORIZER_TYPES, MODEL_CONFIGS.keys()
    ))
    if max_combos:
        combos = combos[:max_combos]

    for field, stop, ngram, max_feat, vtype, model_name in tqdm(combos, desc=dataset_scope):
        text_col = get_text_col(field, stop)
        X_train = train[text_col].fillna('')
        X_val = val[text_col].fillna('')
        y_train, y_val = train['label'], val['label']

        cfg = MODEL_CONFIGS[model_name]
        param_name = cfg['param_name']
        best_f2, best_param, best_metrics = -1, None, None

        for param_val in cfg['params'][param_name]:
            pipe = Pipeline([
                ('vec', build_vectorizer(vtype, ngram, max_feat)),
                ('clf', build_model(model_name, param_val)),
            ])
            pipe.fit(X_train, y_train)
            y_val_pred = pipe.predict(X_val)
            m = compute_metrics(y_val, y_val_pred)
            if m['f2_fake'] > best_f2:
                best_f2 = m['f2_fake']
                best_param = param_val
                best_metrics = m

        row = metrics_to_row(best_metrics, {
            'model': model_name,
            'vectorizer': vtype,
            'text_field': field,
            'stopwords': stop,
            'ngram_range': str(ngram),
            'max_features': max_feat,
            'best_param': best_param,
            'dataset_scope': dataset_scope,
            'split': 'val',
        })
        results.append(row)

    return pd.DataFrame(results)


def build_pipeline_from_config(config_row):
    ngram = eval(config_row['ngram_range'])
    return Pipeline([
        ('vec', build_vectorizer(
            config_row['vectorizer'], ngram, int(config_row['max_features'])
        )),
        ('clf', build_model(config_row['model'], config_row['best_param'])),
    ])


def predict_proba_scores(pipe, X):
    """Scores para ROC/AUC; usa el pipeline completo (vectoriza + clasifica)."""
    clf = pipe.named_steps['clf']
    if hasattr(clf, 'predict_proba'):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(clf, 'decision_function'):
        scores = pipe.decision_function(X)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    return None


def evaluate_on_test(config_row, train, test):
    """Entrena con train y evalúa una sola vez en test."""
    text_col = get_text_col(config_row['text_field'], config_row['stopwords'])
    pipe = build_pipeline_from_config(config_row)
    pipe.fit(train[text_col].fillna(''), train['label'])

    X_test = test[text_col].fillna('')
    y_test_pred = pipe.predict(X_test)
    y_proba = predict_proba_scores(pipe, X_test)

    test_m = compute_metrics(test['label'], y_test_pred, y_proba)
    result_row = metrics_to_row(test_m, {
        'model': config_row['model'],
        'vectorizer': config_row['vectorizer'],
        'text_field': config_row['text_field'],
        'stopwords': config_row['stopwords'],
        'ngram_range': config_row['ngram_range'],
        'max_features': int(config_row['max_features']),
        'best_param': config_row['best_param'],
        'dataset_scope': config_row['dataset_scope'],
        'split': 'test',
    })
    return pipe, result_row

# Experimento principal: subconjunto político (solo métricas de validación)
#politics_results = run_baseline_grid(pol_train, pol_val, 'politics')


In [17]:

# Experimento complementario: dataset completo (solo métricas de validación)
full_results = run_baseline_grid(full_train, full_val, 'full_dataset')

val_results = pd.concat([politics_results, full_results], ignore_index=True)
print('Mejores 10 configuraciones según validación:')
print(val_results.sort_values('f2_fake', ascending=False).head(10).to_string(index=False))


full_dataset:   0%|          | 0/216 [00:00<?, ?it/s]

Mejores 10 configuraciones según validación:
 accuracy  precision_fake  recall_fake  f1_fake  f2_fake  roc_auc               model vectorizer text_field         stopwords ngram_range  max_features  best_param dataset_scope split
 0.996590        0.976871     0.995839 0.986264 0.991987      NaN          linear_svm        bow       body without_stopwords      (1, 2)         10000        10.0  full_dataset   val
 0.994828        0.991908     0.991908 0.991908 0.991908      NaN          linear_svm        bow       full    with_stopwords      (1, 2)         10000         0.1      politics   val
 0.994459        0.990762     0.991908 0.991334 0.991678      NaN logistic_regression        bow       body    with_stopwords      (1, 2)         10000         1.0      politics   val
 0.996249        0.974220     0.995839 0.984911 0.991439      NaN          linear_svm        bow       body    with_stopwords      (1, 2)         10000         1.0  full_dataset   val
 0.996078        0.972900     0.995

## 4. Mejor modelo (selección en val) y evaluación final en test

In [26]:

# Selección final por validación; test solo para el mejor modelo de cada alcance
pol_best_val = politics_results.sort_values('f2_fake', ascending=False).iloc[0]
full_best_val = full_results.sort_values('f2_fake', ascending=False).iloc[0]

pipe_pol, pol_best_test = evaluate_on_test(pol_best_val, pol_train, pol_test)
_, full_best_test = evaluate_on_test(full_best_val, full_train, full_test)

baseline_results = pd.concat([
    val_results,
    pd.DataFrame([pol_best_test, full_best_test]),
], ignore_index=True)
baseline_results.to_csv(RESULTS_METRICS / 'baseline_results.csv', index=False)

print('Mejor configuración politics (val):')
print(pol_best_val.to_string())
print('\nEvaluación en test del mejor modelo politics:')
print(pd.Series(pol_best_test).to_string())

text_col = get_text_col(pol_best_test['text_field'], pol_best_test['stopwords'])
y_pred = pipe_pol.predict(pol_test[text_col].fillna(''))
cm = confusion_matrix(pol_test['label'], y_pred)

plot_confusion_matrix(
    cm, ['Real', 'Fake'],
    f'Matriz de confusión — {pol_best_test["model"]}',
    RESULTS_FIGURES / 'baseline_best_confusion_matrix.png',
)

X_test = pol_test[text_col].fillna('')
y_proba = predict_proba_scores(pipe_pol, X_test)

if y_proba is not None:
    fpr, tpr, _ = roc_curve(pol_test['label'], y_proba)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('Curva ROC — mejor baseline (politics, test)')
    ax.legend()
    save_figure(fig, RESULTS_FIGURES / 'baseline_best_roc_curve.png')
    plt.show()

joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_politics.joblib')
pd.Series(pol_best_test).to_json(RESULTS_MODELS / 'best_baseline_politics_config.json')
joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_model.joblib')
print('Modelos guardados en results/models/')


Mejor configuración politics (val):
accuracy                0.994828
precision_fake          0.991908
recall_fake             0.991908
f1_fake                 0.991908
f2_fake                 0.991908
roc_auc                      NaN
model                 linear_svm
vectorizer                   bow
text_field                  full
stopwords         with_stopwords
ngram_range               (1, 2)
max_features               10000
best_param                   0.1
dataset_scope           politics
split                        val

Evaluación en test del mejor modelo politics:
accuracy                 0.99483
precision_fake          0.994432
recall_fake             0.990022
f1_fake                 0.992222
f2_fake                 0.990901
roc_auc                 0.999249
model                 linear_svm
vectorizer                   bow
text_field                  full
stopwords         with_stopwords
ngram_range               (1, 2)
max_features               10000
best_param                

## 5. Comparación politics vs full_dataset

Si el rendimiento en el dataset completo es mucho mayor que en el subconjunto político, probablemente parte del rendimiento se explica por sesgos de tema, fuente o estructura del dataset — no solo por patrones lingüísticos de fake news.

In [27]:

compare_val = val_results.groupby('dataset_scope')['f2_fake'].max()
compare_test = baseline_results[baseline_results['split'] == 'test'].set_index('dataset_scope')['f2_fake']
print('Mejor F2 en validación por alcance:')
print(compare_val)
print('\nF2 en test del mejor modelo seleccionado en val:')
print(compare_test)

fig, ax = plt.subplots(figsize=(8, 4))
top_pol = politics_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
top_full = full_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
sns.barplot(
    x=['politics (top-5 avg)', 'full_dataset (top-5 avg)'],
    y=[top_pol, top_full],    ax=ax,
)
ax.set_ylabel('F2 fake en validación (promedio top-5)')
ax.set_title('Comparación subconjunto político vs dataset completo (val)')
save_figure(fig, RESULTS_FIGURES / 'baseline_politics_vs_full.png')
plt.show()


Mejor F2 en validación por alcance:
dataset_scope
full_dataset    0.991987
politics        0.991908
Name: f2_fake, dtype: float64

F2 en test del mejor modelo seleccionado en val:
dataset_scope
politics        0.990901
full_dataset    0.990202
Name: f2_fake, dtype: float64


## Conclusiones

- Se compararon LR, MNB y Linear SVM con BoW y TF-IDF.
- Se evaluaron título, cuerpo y título+cuerpo; con y sin stopwords.
- La métrica principal F2 prioriza detectar fake news (minimizar falsos negativos).
- El modelo aprende patrones del dataset, no verifica hechos.